# VICReg Pre-training for Ultrasound ViT

Trains a **VICReg** (Variance-Invariance-Covariance Regularization) ViT encoder
on grayscale ultrasound images and plots training metrics live.

**Reference:** Bardes et al., "VICReg: Variance-Invariance-Covariance Regularization
for Self-Supervised Learning", ICLR 2022.

## Workflow
1. Load dataset
2. Configure model & training hyperparameters
3. **Train** with `train_vicreg` — loss breakdown + RankME plots each epoch
4. Extract embeddings / run linear probe

In [4]:
!ls /content

datasets_adapters  embeddings  sample_data


In [3]:
%matplotlib inline

import math, os, sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

PROJECT_ROOT = str(Path.cwd().resolve().parents[0])  # two levels up from embeddings/vicreg/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from embeddings.vicreg.train import train_vicreg, load_checkpoint

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Project root: /
PyTorch: 2.10.0+cu128
CUDA available: True


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load Dataset

In [ ]:
from datasets_adapters.fetal_planes_db.fpd_dataset import FetalPlanesDBDataset
from datasets_adapters.fetal_head_circ.fhc_dataset import FetalHeadCircDataset
from datasets_adapters.psfhs_head.psfhs_dataset import PSFHSDataset
from datasets_adapters.focus import FOCUSDataset
from datasets_adapters.abdominal_segmentation import AbdominalSegmentationDataset
from torch.utils.data import ConcatDataset

DATASET_ROOT  = "/content/drive/MyDrive/SHAD/project/fetal_planes_db"
FHC_IMAGES    = "/content/drive/MyDrive/SHAD/project/fetal_head_circumference/training_set/training_set"
FHC_CSV       = "/content/drive/MyDrive/SHAD/project/fetal_head_circumference/training_set_pixel_size_and_HC.csv"
FOCUS_ROOT    = "/content/drive/MyDrive/SHAD/project/focus"
ABDOMINAL_ROOT = "/content/drive/MyDrive/SHAD/project/FetalAbdominalSegmentationDataset"
PSFHS_ROOT = "/content/drive/MyDrive/SHAD/project/PSFHS"


dataset = ConcatDataset([
    FetalPlanesDBDataset(root=DATASET_ROOT, images_dir='Images'),
    FetalHeadCircDataset(images_dir=FHC_IMAGES, csv_file=FHC_CSV, load_annotations=False),
    FOCUSDataset(root=FOCUS_ROOT, split="training", load_masks=False),
    AbdominalSegmentationDataset(root=ABDOMINAL_ROOT, load_masks=False),
    PSFHSDataset(root=PSFHS_ROOT, load_masks=False),
])
print(f"Dataset size: {len(dataset)} images")

Loaded 12400 images from /content/drive/MyDrive/SHAD/project/fetal_planes_db
Loaded 999 images from /content/drive/MyDrive/SHAD/project/fetal_head_circumference/training_set/training_set
Loaded 200 images from /content/drive/MyDrive/SHAD/project/focus/training
Loaded 1588 samples from /content/drive/MyDrive/SHAD/project/FetalAbdominalSegmentationDataset/ARRAY_FORMAT
Dataset size: 15187 images


## 2. Configuration

In [13]:
# ── Model ──────────────────────────────────────────────────────────
MAX_IMAGE_HEIGHT = 224
PATCH_SIZE       = 16
EMBED_DIM        = 512
DEPTH            = 12
NUM_HEADS        = 8
MLP_RATIO        = 4.0
PROJECTOR_DIM    = 2048   # projector output dim; 4× encoder dim is typical
PROJECTOR_LAYERS = 3

# ── VICReg loss ────────────────────────────────────────────────────
SIM_COEFF  = 25.0   # invariance (λ in paper)
STD_COEFF  = 25.0   # variance hinge (μ in paper)
COV_COEFF  = 1.0    # covariance regularization (ν in paper)

# ── Training ──────────────────────────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 256   # larger batches improve covariance estimates
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 10
MIN_LR        = 1e-6
VAL_SPLIT     = 0.1
NUM_WORKERS   = 8

# ── Checkpointing ─────────────────────────────────────────────────
CHECKPOINT_DIR = os.path.join(
    '/content/drive/MyDrive/SHAD/project', 'checkpoints', 'vicreg', 'v3.2'
)
SAVE_EVERY    = 20
PLOT_EVERY = 5
RANKME_EVERY  = 5    # compute RankME every N epochs; 0 to disable
PREDEFINED_WEIGHTS = 'vit_mediumd_patch16_rope_reg1_gap_256'

# ── Optional: warm-start encoder from a pre-trained MAE checkpoint ─
MAE_INIT_CHECKPOINT = None
# MAE_INIT_CHECKPOINT = os.path.join(..., 'mae_final.pt')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 3. Train

The callback redraws the dashboard each epoch:
- **Total loss** (train/val)
- **Loss components** (invariance / variance / covariance)
- **RankME** (when `RANKME_EVERY > 0`)
- **LR schedule**

In [14]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def update_plots(history: dict, epochs_total: int) -> None:
    clear_output(wait=True)
    e = len(history["train_loss"])
    xs = range(1, e + 1)

    has_rankme = any(v is not None for v in history.get("val_rank_me", []))
    ncols = 4 if has_rankme else 3
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))

    # Total loss
    ax = axes[0]
    ax.plot(xs, history["train_loss"], label="train", linewidth=1.5)
    ax.plot(xs, history["val_loss"],   label="val",   linewidth=1.5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Total VICReg Loss")
    ax.legend(); ax.grid(True, alpha=0.3)

    # Loss components (training)
    ax = axes[1]
    ax.plot(xs, history["train_loss_sim"], label="sim (inv)",  linewidth=1.5)
    ax.plot(xs, history["train_loss_var"], label="var (hinge)",linewidth=1.5)
    ax.plot(xs, history["train_loss_cov"], label="cov (decor)",linewidth=1.5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Raw term value")
    ax.set_title("Loss Components (train, unweighted)")
    ax.legend(); ax.grid(True, alpha=0.3)

    # LR
    ax = axes[2]
    ax.plot(xs, history["lr"], color="tab:orange", linewidth=1.5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("LR")
    ax.set_title("LR Schedule")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(True, alpha=0.3)

    # RankME
    if has_rankme:
        ax = axes[3]
        rm_xs = [i + 1 for i, v in enumerate(history["val_rank_me"]) if v is not None]
        rm_tr = [v for v in history["train_rank_me"] if v is not None]
        rm_val = [v for v in history["val_rank_me"] if v is not None]
        ax.plot(rm_xs, rm_tr,  "o-", label="train", linewidth=1.5, markersize=4)
        ax.plot(rm_xs, rm_val, "o-", label="val",   linewidth=1.5, markersize=4)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Effective rank")
        ax.set_title("RankME")
        ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    last_rm = next((v for v in reversed(history["val_rank_me"]) if v is not None), None)
    rm_str = f"  rank_me={last_rm:.1f}" if last_rm is not None else ""
    print(
        f"Epoch {e}/{epochs_total}  |  "
        f"loss={history['train_loss'][-1]:.4f}/{history['val_loss'][-1]:.4f}  |  "
        f"sim={history['train_loss_sim'][-1]:.3f}  "
        f"var={history['train_loss_var'][-1]:.3f}  "
        f"cov={history['train_loss_cov'][-1]:.3f}  |  "
        f"lr={history['lr'][-1]:.2e}"
        + rm_str
    )


def on_epoch_end(epoch, history, model):
    update_plots(history, EPOCHS)

In [16]:
model, history = train_vicreg(
    dataset,
    max_image_height=MAX_IMAGE_HEIGHT,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    projector_dim=PROJECTOR_DIM,
    projector_layers=PROJECTOR_LAYERS,
    sim_coeff=SIM_COEFF,
    std_coeff=STD_COEFF,
    cov_coeff=COV_COEFF,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    min_lr=MIN_LR,
    val_split=VAL_SPLIT,
    num_workers=NUM_WORKERS,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
    rankme_every=RANKME_EVERY,
    mae_init_checkpoint=MAE_INIT_CHECKPOINT,
    device=DEVICE,
    use_amp=True,
    on_epoch_end=on_epoch_end,
    plot_every=PLOT_EVERY,
    # pretrained_encoder=PREDEFINED_WEIGHTS,
    timm_encoder=PREDEFINED_WEIGHTS,
)

Epoch 35/100  |  loss=15.0897/15.4853  |  sim=0.047  var=0.440  cov=2.900  |  lr=2.51e-04  rank_me=270.5
  → training plots saved: /content/drive/MyDrive/SHAD/project/checkpoints/vicreg/v3.2/training_plots.png


Epoch 35 [train]:  77%|███████▋  | 41/53 [00:36<00:13,  1.15s/it, loss=15.092, sim=0.050, var=0.437]

: 

## 4. Resume Training (optional)

In [7]:
RESUME_FROM = os.path.join(CHECKPOINT_DIR, "vicreg_epoch_50.pt")

model, history = train_vicreg(
    dataset,
    max_image_height=MAX_IMAGE_HEIGHT,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    projector_dim=PROJECTOR_DIM,
    projector_layers=PROJECTOR_LAYERS,
    sim_coeff=SIM_COEFF,
    std_coeff=STD_COEFF,
    cov_coeff=COV_COEFF,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    min_lr=MIN_LR,
    val_split=VAL_SPLIT,
    num_workers=NUM_WORKERS,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every=SAVE_EVERY,
    rankme_every=RANKME_EVERY,
    device=DEVICE,
    use_amp=True,
    on_epoch_end=on_epoch_end,
    resume_from=RESUME_FROM,
)

Training on: cuda
Train: 13669 samples | Val: 1518 samples
Parameters: 21,393,408 encoder | 9,183,232 projector | 30,576,640 total trainable
Mixed precision (AMP) enabled.


/content/embeddings/vicreg/train.py:658: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if _use_amp else None


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SHAD/project/checkpoints/vicreg/v3.1/vicreg_epoch_50.pt'

## 5. Extract Embeddings & RankME

In [ ]:
from torch.utils.data import DataLoader
from functools import partial
from embeddings.vit.train import MAEValTransform, _MAEDatasetWrapper, mae_pad_collate
from embeddings.rank_me import compute_rank_me, collect_embeddings

# Build a simple single-view loader over the full dataset
_transform = MAEValTransform(max_image_height=MAX_IMAGE_HEIGHT)
_ds = _MAEDatasetWrapper(dataset, _transform)
_loader = DataLoader(
    _ds, batch_size=256, shuffle=False, num_workers=4,
    collate_fn=partial(mae_pad_collate, patch_size=PATCH_SIZE),
)

model.eval()
embs = collect_embeddings(model, _loader, DEVICE)
rank_me = compute_rank_me(embs)

print(f"Embeddings shape: {embs.shape}")
print(f"RankME (full dataset): {rank_me:.2f}  (max possible = {embs.shape[1]})")

## 6. Linear Probe Evaluation

In [ ]:
from datasets_adapters.fetal_planes_db.linear_probe import train_linear_probe

probe_history = train_linear_probe(
    encoder=model,          # VICRegModel has .encode() — compatible with linear_probe
    embed_dim=EMBED_DIM,
    dataset_root=DATASET_ROOT,
    images_dir="Images",
    max_image_height=MAX_IMAGE_HEIGHT,
    epochs=50,
    batch_size=128,
    lr=1e-3,
    weight_decay=1e-4,
    device=DEVICE,
    checkpoint_dir=os.path.join(CHECKPOINT_DIR, "linear_probe"),
    save_every=10,
    num_workers=8,
    show_plot=True,
)
print(f"Best val accuracy: {100 * max(probe_history['val_acc']):.2f}%")